## Mount Google Drive and Set Project Directory
This cell mounts your Google Drive to `/content/drive` and then sets the current working directory to the specified `BASE_PROJECT_DIR`. This ensures all subsequent file paths are relative to your project's root folder on Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Define the base directory for the project.
# All subsequent file paths will be relative to this directory.
BASE_PROJECT_DIR = "your/project/path"
os.chdir(BASE_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import Libraries
This cell imports essential Python libraries for data manipulation, deep learning (TensorFlow and Keras), and custom data loading utilities (`custom_datagen`).

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import MeanIoU
import pandas as pd

import custom_datagen

## Define Test Data Paths and List Files
This cell defines the directories for test images and masks. It then uses `os.listdir` and `sorted` to create sorted lists of filenames within these directories and prints the total number of images and masks found.

In [3]:
# The paths are now relative to the current working directory (BASE_PROJECT_DIR).
test_img_dir  = "BraTS2020_TrainingData/input_data_128/val/images/"
test_mask_dir = "BraTS2020_TrainingData/input_data_128/val/masks/"

test_img_list  = sorted(os.listdir(test_img_dir))
test_mask_list = sorted(os.listdir(test_mask_dir))

print("Num test images:", len(test_img_list))
print("Num test masks :", len(test_mask_list))

Num test images: 69
Num test masks : 69


## Define Model Paths
This cell defines a list of paths to the Keras/HDF5 model files that will be evaluated. These paths are relative to the `BASE_PROJECT_DIR` set earlier.

In [4]:
# The model paths are now relative to the current working directory (BASE_PROJECT_DIR).
model_paths = [
    "AE-UVNet.keras",
    "BottleneckVnet.hdf5",
    "VnetNoBottleneck.hdf5",
    "BottleneckUNet.hdf5",
    "UNetNoBottleneck.hdf5"
]

## Define General Dice Score Function
This cell defines a utility function `dice_score` that calculates the Dice Similarity Coefficient (DSC) between two binary arrays (`y_true` and `y_pred`). It flattens the arrays and computes the intersection over the sum of their elements.

In [5]:
def dice_score(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

## Define Binary Dice Score Function for Tumor Regions
This cell defines a `dice_score_binary` function, which is identical to `dice_score`. It is specifically intended for calculating Dice scores for different tumor regions (Whole Tumor, Tumor Core, Enhancing Tumor) by comparing binary masks.

In [6]:
# Dice Scores for WT,ET and TC
def dice_score_binary(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

## Evaluate Models and Calculate Dice Scores
This crucial cell iterates through each model defined in `model_paths`. For each model, it:
1.  Loads the model.
2.  Initializes a data generator to load test images and masks.
3.  Performs prediction on a batch of test images.
4.  Converts ground truth and predicted masks to argmax and then to one-hot encoded formats.
5.  Extracts binary masks for Whole Tumor (WT), Tumor Core (TC), and Enhancing Tumor (ET) from both true and predicted masks.
6.  Calculates and prints the Dice scores for WT, TC, and ET using the `dice_score_binary` function.
7.  Prints the mean Dice score across these three regions.

In [7]:
from tensorflow.keras.utils import to_categorical
for my_model in model_paths:
    print(my_model)
    my_model = os.path.join(BASE_PROJECT_DIR+"/Model/",my_model)
    my_model = load_model(my_model, compile=False)
    batch_size = 1
    test_img_datagen = custom_datagen.imageLoader(test_img_dir, test_img_list,
                                               test_mask_dir, test_mask_list, batch_size)
    #Verify the generator
    test_image_batch, test_mask_batch = test_img_datagen.__next__()

    test_mask_batch_argmax = np.argmax(test_mask_batch, axis=4)
    test_pred_batch = my_model.predict(test_image_batch)
    test_pred_batch_argmax = np.argmax(test_pred_batch, axis=4)

    # Convert predictions and masks to one-hot
    y_true = to_categorical(test_mask_batch_argmax, num_classes=4)
    y_pred = to_categorical(test_pred_batch_argmax, num_classes=4)


    true_mask_argmax = test_mask_batch_argmax
    pred_mask_argmax = test_pred_batch_argmax

    # Whole Tumor (WT): 1, 2, 3
    true_wt = np.isin(true_mask_argmax, [1, 2, 3]).astype(np.uint8)
    pred_wt = np.isin(pred_mask_argmax, [1, 2, 3]).astype(np.uint8)

    # Tumor Core (TC): 1, 3
    true_tc = np.isin(true_mask_argmax, [1, 3]).astype(np.uint8)
    pred_tc = np.isin(pred_mask_argmax, [1, 3]).astype(np.uint8)

    # Enhancing Tumor (ET): 3
    true_et = (true_mask_argmax == 3).astype(np.uint8)
    pred_et = (pred_mask_argmax == 3).astype(np.uint8)
    wt_dice = dice_score_binary(true_wt, pred_wt)
    tc_dice = dice_score_binary(true_tc, pred_tc)
    et_dice = dice_score_binary(true_et, pred_et)

    print(f"Whole Tumor Dice (WT): {wt_dice:.4f}")
    print(f"Tumor Core Dice (TC): {tc_dice:.4f}")
    print(f"Enhancing Tumor Dice (ET): {et_dice:.4f}")
    print(f"Mean Dice: {(wt_dice+tc_dice+et_dice)/3:.4f}")


AE-UVNet.keras
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Whole Tumor Dice (WT): 0.9358
Tumor Core Dice (TC): 0.8085
Enhancing Tumor Dice (ET): 0.8136
Mean Dice: 0.8526
BottleneckVnet.hdf5
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Whole Tumor Dice (WT): 0.9172
Tumor Core Dice (TC): 0.5301
Enhancing Tumor Dice (ET): 0.7488
Mean Dice: 0.7321
VnetNoBottleneck.hdf5
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Whole Tumor Dice (WT): 0.8817
Tumor Core Dice (TC): 0.6747
Enhancing Tumor Dice (ET): 0.7276
Mean Dice: 0.7613
BottleneckUNet.hdf5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Whole Tumor Dice (WT): 0.9132
Tumor Core Dice (TC): 0.6910
Enhancing Tumor Dice (ET): 0.7774
Mean Dice: 0.7939
UNetNoBottleneck.hdf5


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Whole Tumor Dice (WT): 0.9002
Tumor Core Dice (TC): 0.6686
Enhancing Tumor Dice (ET): 0.7836
Mean Dice: 0.7841
